# Data Representation or Vectorization

Some common terms to remember:
- Corpus
- Vocabulary
- Document
- Word

# One Hot Encoding

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
sentences = [
    "People watch MrBeast",
    "MrBeast watch MrBeast",
    "People buy Beast chocolates",
    "MrBeast buy Beast chocolates"
]

vocabulary = set()
for sentence in sentences:
    words = sentence.split()
    for word in words:
        vocabulary.add(word)

vocabulary = sorted(list(vocabulary))  # ['Beast', 'MrBeast', 'People', 'buy', 'chocolates', 'watch']

# Create one-hot encoding
one_hot_encoded = []
for sentence in sentences:
    words = sentence.split()
    encoding = np.zeros(len(vocabulary))
    for word in words:
        encoding[vocabulary.index(word)] = 1
    one_hot_encoded.append(encoding)

# Create DataFrame
df = pd.DataFrame(one_hot_encoded, columns=vocabulary)
df

,Beast,MrBeast,People,buy,chocolates,watch
0,0.0,1.0,1.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,0.0,1.0
2,1.0,0.0,1.0,1.0,1.0,0.0
3,1.0,1.0,0.0,1.0,1.0,0.0


# Bag of Words

In [ ]:
df = pd.DataFrame({
    "Text": [
        "People watch MrBeast",
        "MrBeast watch MrBeast",
        "People buy Beast chocolates",
        "MrBeast buy Beast chocolates"
    ], "+/-":[1, 1, 0, 0]
})

df

,Text,+/-
0,People watch MrBeast,1
1,MrBeast watch MrBeast,1
2,People buy Beast chocolates,0
3,MrBeast buy Beast chocolates,0


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [6]:
bow = cv.fit_transform(df['Text'])

In [7]:
print(cv.vocabulary_)

{'people': 4, 'watch': 5, 'mrbeast': 3, 'buy': 1, 'beast': 0, 'chocolates': 2}


In [8]:
print(bow.toarray())

[[0 0 0 1 1 1]
 [0 0 0 2 0 1]
 [1 1 1 0 1 0]
 [1 1 1 1 0 0]]


# Word2vec Implementation

Using Game of Thrones dataset from Kaggle.

Link: https://drive.google.com/file/d/1lbtAwzE7l0otXYFDtGUKKWzI83bD5D5H/view?usp=drive_link

In [11]:
import gensim
import os

In [12]:
!pip install --upgrade gensim --user

In [18]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [19]:
story = []
for filename in os.listdir('data'):
    if filename == '.ipynb_checkpoints':
      pass
    f = open(os.path.join('data',filename))
    corpus = f.read()
    raw_sent = sent_tokenize(corpus)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [ ]:
story

In [22]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

model.build_vocab(story)

model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(322801, 447775)

In [23]:
model.wv.most_similar('daenerys')

[('still', 0.9992486238479614),
 ('all', 0.9991886615753174),
 ('man', 0.9991364479064941),
 ('other', 0.9991334676742554),
 ('illyrio', 0.9991301894187927),
 ('prince', 0.999129056930542),
 ('three', 0.9991236329078674),
 ('two', 0.9991233348846436),
 ('an', 0.9991230964660645),
 ('robb', 0.9991018772125244)]

In [24]:
model.wv.similarity('arya','sansa')

0.9997294

In [25]:
model.wv['deep'].shape

(100,)

In [26]:
vec = model.wv.get_normed_vectors()

In [27]:
vec

array([[-0.1317164 ,  0.06140587,  0.10478807, ..., -0.09423025,
         0.09191024,  0.02411414],
       [-0.12644698,  0.06000207,  0.09578862, ..., -0.09179283,
         0.08786353,  0.0131291 ],
       [-0.09127679,  0.04405374,  0.07446842, ..., -0.08421682,
         0.08285073, -0.01498588],
       ...,
       [-0.07673036,  0.10316883,  0.10255229, ..., -0.10461285,
         0.08404985,  0.02323096],
       [-0.084352  ,  0.07566347,  0.05113149, ..., -0.11190403,
         0.06336261, -0.02925051],
       [-0.026114  ,  0.10952298,  0.00433726, ..., -0.07220009,
         0.09287413, -0.0171342 ]], dtype=float32)

In [29]:
model.wv.get_normed_vectors().shape

(3840, 100)

In [30]:
y = model.wv.index_to_key

In [31]:
len(y)

3840

In [ ]:
y

In [33]:
from sklearn.decomposition import PCA
pca = PCA(n_components=3)
X = pca.fit_transform(model.wv.get_normed_vectors())

In [34]:
X

array([[-0.00862783, -0.20386945, -0.0057307 ],
       [-0.02409983, -0.15472783, -0.00124568],
       [-0.05533868,  0.02438318,  0.0018713 ],
       ...,
       [ 0.00930625,  0.06038818, -0.03744026],
       [-0.0173564 , -0.05385992,  0.01534638],
       [ 0.03203541,  0.09629954,  0.09028383]], dtype=float32)

In [35]:
X.shape

(3840, 3)

In [36]:
import plotly.express as px
fig = px.scatter_3d(X[200:300],x=0,y=1,z=2, color=y[200:300])
fig.show()